In [2]:
import sys
sys.path.append(r"C:\Users\Sasmit\Desktop\Courses\3rd Year\Machine Design\RM_Python_Lib\rm_python_lib")



In [3]:
# import statements
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

import MechDesign.Helpers as HM

from MechDesign.Units.Units import m_, mm_, kg_, s_, N_, rpm_, W_, deg_
import MechDesign.Units.UnitMethods as UM

import MechDesign.RnM as RnM

C:\Users\Sasmit\Desktop\Courses\3rd Year\Machine Design\RM_Python_Lib\rm_python_lib\MechDesign\RnM\C3_StrengthAndStress.py:36: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
  'K_alpha'           :'[] Shape-dependent size factor of the part and test rod respectively,\ (see 3.5.1-3.), values according to table 3-11d' ,\
C:\Users\Sasmit\Desktop\Courses\3rd Year\Machine Design\RM_Python_Lib\rm_python_lib\MechDesign\RnM\C3_StrengthAndStress.py:37: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
  'K_alphaTest'       :'[] Shape-dependent size factor of the part and test rod respectively,\ (see 3.5.1-3.), values according to table 3-11d' ,\


In [ ]:
# creating the chain to be calculated
CH = RnM.Chain()     #--- since Chain is part of the RnM library we call it as a part of it ---

In [7]:
final_shaft_speed_kmh = 15.5

final_shaft_speed_ms = final_shaft_speed_kmh * 1000 / 3600
print (final_shaft_speed_ms)

final_shaft_rpm = final_shaft_speed_ms / (2 * np.pi * (0.550/2)) * 60
print (final_shaft_rpm)

4.305555555555555
149.50918896511376


In [12]:
from sympy import symbols, Eq, solve

motor_shaft_rpm = 1500

i_total = motor_shaft_rpm / final_shaft_rpm
print (i_total)

i_1, i_2 = symbols( 'i_1 i_2', real=True, positive=True)
sol = solve( [Eq(i_1 * i_2, i_total), Eq(i_1/i_2, 0.6)], (i_1, i_2))

print (sol)

i_1 = sol[0][0]
i_2 = sol[0][1]
print (i_1)
print (i_2)

10.032828151786761
[(2.45350705951136, 4.08917843251893)]
2.45350705951136
4.08917843251893


In [13]:
middle_shaft_rpm = motor_shaft_rpm / i_1
print (middle_shaft_rpm)

611.369750979539


In [18]:
eta_belt = 0.96
eta_gear = 0.98
eta_bearing = 0.99
eta_coupling = 0.99

eta_total = eta_belt * eta_gear * (eta_bearing**4) * eta_coupling
print (eta_total)

0.89469143894592


In [20]:
P_output = 8000
 
P_input = P_output / eta_total 
print (P_input)

8941.630210997877


In [22]:
diameter_pulley_motor = 131.3 * mm_
diameter_pulley_middle = (diameter_pulley_motor * i_1) * mm_

print (diameter_pulley_motor)
print (diameter_pulley_middle)

131.3*mm_
322.145476913842*mm_**2


In [ ]:
K_A = 2.2
required_motor_power = P_input * K_A

print (required_motor_power)

nominal_motor_power = 22000

selected_motor = "180-L"

print ("The selected motor is:", selected_motor)

19671.58646419533
The selected motor is: 180-L


In [30]:
# creating a new shaft connection, (this is 1 specific connection in your design)
SC = RnM.ShaftConnection() 

SC.p_gem = SC.E12_1B_KeyAveragePressure()  # this specific instance of SC.p_gem now hold the expression, with all of the symbols
HM.EqPrint('p_average',SC.p_gem) # as can be seen, the displayed formula is identical, except the left hand
dummy=HM.MyHelp(SC.p_gem)  # the explenations found are the default explenations

Eq(p_average, 2*_K_lambda*_T_eq/(_d*_hprime*_lprime*_n*_phi))

2*_K_lambda*_T_eq/(_d*_hprime*_lprime*_n*_phi)

    d : [mm] diameter of the shaft for splines: {tabel 12-3a} or addition for weakend section due to press fit screws (multiple ring press fit)
    lprime : [mm] effective length of the key
    n : [] number of keys or number of teeth of a spline
    phi : [] correction factor based on number of keys
    K_lambda : [] load distribution factor method C: K_lambda = 1;  method B: {tabel 12-2c}
    hprime : [mm] h' is the effective height 
    T_eq : [Nm] equivalent transmitted torque


In [31]:
SC.d = 48 * mm_

motor_shaft_rads = motor_shaft_rpm * 2 * np.pi / 60

SC.T_nom = (P_input / motor_shaft_rads) * N_*m_
SC.K_A = 2.2

# setting up constants that are not likely to change
SC.phi = 1 # 1key
SC.n = 1 # 1 key
SC.K_lambda = 1 # method C

SC.b = 14 * mm_
SC.h = 9 * mm_
SC.l = 50 * mm_ # Change this value

SC.lprime = SC.E12_1_hI_KeyEffectiveLength()
SC.hprime = SC.E12_1_hJ_KeyEffectiveHeight()
SC.T_eq = SC.E12_1_hC_DynamicLoadTorque()



In [32]:
# in order to replace the symbols with their value counterparts we have to substitute the symbols in the expression of  p_gem with the specific values (or expressions) from a specific shaft connection instance. 
# broken down into different steps:

# 1 copy expression into a temporary expression
MyExp = SC.p_gem
HM.EqPrint('MyExpOnP_gem',MyExp)
# 2 substitute the symbols in this temporary expression with the values, symbols or expressions from a specific shaft connection (SC)
MyExp2 = HM.substitute(MyExp,SC)
HM.EqPrint('MyEx_2_pOnP_gem',MyExp2)

# intermezzo: the 'substitute' function can also be used to substitute a specific symbol with something else. 
# pairs of symbol and value have to be provided, separated with a ',' 
# e.g.: setting the diameter to 2*60 and K_lambda to 2, will cancel out. Resulting in the identical value for p_gem
# Non-specified parameters will be taken from 'SC'
# Do experiment with some other values, in order to get accustomed with this function.
MyExp3 = HM.substitute(MyExp,SC,d=2*60*mm_,K_lambda=2)  
HM.EqPrint('MyEx_3_pOnP_gem',MyExp3)

# 3 replace the expression for p_gem, in a specific shaft connection, with the new expression
SC.p_gem = MyExp2

# or short:
# SC.p_gem = MD.substitute(SC.p_gem,SC)

# as can be seen from the display statement, units are still to be simplified. 
SC.p_gem = UM.m_to_mm(SC.p_gem)
t=HM.EqPrint('p_gem',SC.p_gem)

Eq(MyExpOnP_gem, 2*_K_lambda*_T_eq/(_d*_hprime*_lprime*_n*_phi))

Eq(MyEx_2_pOnP_gem, 0.03579*N_*m_/mm_**3)

Eq(MyEx_3_pOnP_gem, 0.02863*N_*m_/mm_**3)

Eq(p_gem, 35.79*N_/mm_**2)

In [40]:
# setting the value of known parameters
CH.n_1= middle_shaft_rpm * rpm_    
CH.n_2= final_shaft_rpm * rpm_
t=HM.EqPrint('n_1',CH.n_1)     

Eq(n_1, 611.4*rpm_)

In [41]:
# An initial estimate of the gear ratio is needed; the final gear ratio will depend on the number of teeth on both gear wheels.
iprime1 = CH.n_1/CH.n_2
t=HM.EqPrint('iprime1',iprime1)       

Eq(iprime1, 4.089)

In [43]:
CH.z_1 = 23
CH.z_2 = 94

CH.i = CH.E17_1B_GearRatioTeeth() 
t=HM.EqPrint('i',CH.i*mm_)

Eq(i, 4.087*mm_)